<a href="https://colab.research.google.com/github/HowardChen0211/dataManPy/blob/main/ProblemSet_3_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

# California Housing Market, 2020–2025: Home Values, Rents, and Local Labor Markets

### Research questions
1. Which California counties saw the fastest growth in home values from 2020 to 2025?
2. To what extent is home-value growth associated with rent growth across counties?
3. Are county size (population) and the unemployment rate related to how fast housing costs grew?


### Why these data
I use Zillow housing data together with county-level population and labor-market data. The Zillow data provide measures of home values and rents, while population and labor-market data provide additional information about the characteristics of each county.

These datasets are useful together because they allow me to move beyond describing housing prices alone and begin examining how housing markets differ across places with different demographic and economic characteristics.

#### Unit of analysis
California counties (n = 58). The merge key across all datasets is the **FIPS code**.


## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#will display all output not just last command
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

pd.set_option('display.float_format', '{:.3f}'.format) # Turn off the scientific notation https://medium.com/@amit25173/steps-to-pandas-turn-off-scientific-notation-62c44b86ee24

# ProblemSet 0

## Load the data

In [2]:
# Zillow Home Value Index (ZHVI)
zhvi = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")

# Zillow Observed Rent Index (ZORI)
zori = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/County_zori_uc_sfrcondomfr_sm_month.csv")

# County population 2025
population = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/CA_Population_estimates_2025.csv")

# ACS 2020-2024 5-year: civilian labor force 16 and over,
unemployment = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/CA_UnemploymentRate.csv")

In [3]:
# take a look about the data (first 5 rows)
zhvi.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2000-01-31,...,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,210217.606,...,876483.587,880007.554,881688.904,881664.430,879853.110,877155.644,874790.007,872663.237,872220.754,873008.422
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,152726.258,...,327258.568,328895.004,330363.857,332095.628,333712.156,335053.112,336017.428,337263.419,339162.650,341569.751
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,113021.492,...,283353.950,283400.843,283252.913,282917.698,282310.099,281596.697,280609.690,279554.676,278797.692,278517.967
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,146106.504,...,459582.164,460367.020,461220.421,461894.031,461740.422,460735.658,459147.558,457589.894,456733.110,456252.051
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,216416.160,...,925424.341,927598.249,928828.819,929848.898,930228.330,930115.693,929543.266,928860.114,929330.518,930799.655


In [4]:
# take a look about the data (first 5 rows)
zori.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2015-01-31,...,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,1691.320,...,2777.262,2768.420,2770.406,2776.600,2788.838,2797.135,2805.344,2809.822,2810.309,2813.614
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,1413.147,...,2145.212,2143.577,2157.834,2178.380,2204.530,2226.641,2248.019,2265.658,2274.611,2272.117
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,1171.144,...,1577.797,1572.948,1571.596,1569.418,1570.920,1573.802,1580.332,1584.251,1586.397,1586.095
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,905.494,...,1692.043,1689.654,1690.670,1697.466,1702.306,1706.189,1711.234,1712.111,1715.836,1716.663
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,1628.319,...,2917.657,2912.131,2915.513,2929.541,2941.689,2951.787,2963.444,2976.321,2984.436,2993.632


In [5]:
# take a look about the data (first 5 rows)
population.head()

,FIPS Code,Summary Level,Area Name,Qualified Area Name,"Total Population (July 1, 2025)"
0,Geo_FIPS,_geo_level_,Geo_NAME,Geo_qname,ORG_T007_001
1,06001,SL050,Alameda County,"Alameda County, California",1636630
2,06003,SL050,Alpine County,"Alpine County, California",1043
3,06005,SL050,Amador County,"Amador County, California",41876
4,06007,SL050,Butte County,"Butte County, California",209211


In [6]:
# take a look about the data (first 5 rows)
unemployment.head()

,FIPS,GeoLevel,NAME,Qualified Area Name,Area (Land),Area (Water),Civilian Population in Labor Force 16 Years and Over,Civilian Population in Labor Force 16 Years and Over: Employed,Civilian Population in Labor Force 16 Years and Over: Unemployed
0,Geo_FIPS,_geo_level_,Geo_NAME,Geo_qname,Geo_AREALAND,Geo_AREAWATER,SE_A17005_001,SE_A17005_002,SE_A17005_003
1,06001,SL050,"Alameda County, California","Alameda County, California",1910010500,216909647,908995,860129,48866
2,06003,SL050,"Alpine County, California","Alpine County, California",1912292607,12557304,837,800,37
3,06005,SL050,"Amador County, California","Amador County, California",1539967082,29437117,17257,16325,932
4,06007,SL050,"Butte County, California","Butte County, California",4238545149,105204062,100743,93228,7515


**Interpretation**: Both Zillow files are national and wide, with one row per U.S. county and one column per month, so I have to filter to California and reshape them before doing anything else.

# ProblemSet 1

#### Data Inspection

In [7]:
# prints information about a DataFrame
print(zhvi.info(), "\n")
print(zori.info(), "\n")
print(population.info(), "\n")
print(unemployment.info(), "\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3071 entries, 0 to 3070
Columns: 329 entries, RegionID to 2026-08-31
dtypes: float64(320), int64(4), object(5)
memory usage: 7.7+ MB
None 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1407 entries, 0 to 1406
Columns: 149 entries, RegionID to 2026-08-31
dtypes: float64(140), int64(4), object(5)
memory usage: 1.6+ MB
None 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 5 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   FIPS Code                        59 non-null     object
 1   Summary Level                    59 non-null     object
 2   Area Name                        59 non-null     object
 3   Qualified Area Name              59 non-null     object
 4   Total Population (July 1, 2025)  59 non-null     object
dtypes: object(5)
memory usage: 2.4+ KB
None 

<class 'pandas.core.frame.DataFrame'>
RangeI

In [8]:
# See how many columns in the data
print("Home Index: ", len(zhvi.columns))
print("Rent Index: ", len(zori.columns))
print("Population: ", len(population.columns))
print("Unemployment: ", len(unemployment.columns))

Home Index:  329
Rent Index:  149
Population:  5
Unemployment:  9


In [9]:
# See how big the data
print(zhvi.shape) # (3071, 329)
print(zori.shape) # (1407, 149)
print(population.shape) # (59, 5)
print(unemployment.shape) # (59, 9)

(3071, 329)
(1407, 149)
(59, 5)
(59, 9)


In [10]:
# Missing values
print(zhvi.isna().sum())
print()
print(zori.isna().sum())
print()
print(population.isna().sum())
print()
print(unemployment.isna().sum())

RegionID      0
SizeRank      0
RegionName    0
RegionType    0
StateName     0
             ..
2026-04-30    0
2026-05-31    0
2026-06-30    0
2026-07-31    0
2026-08-31    0
Length: 329, dtype: int64

RegionID        0
SizeRank        0
RegionName      0
RegionType      0
StateName       0
             ... 
2026-04-30    224
2026-05-31    196
2026-06-30    169
2026-07-31    114
2026-08-31      1
Length: 149, dtype: int64

FIPS Code                          0
Summary Level                      0
Area Name                          0
Qualified Area Name                0
Total Population (July 1, 2025)    0
dtype: int64

FIPS                                                                0
GeoLevel                                                            0
NAME                                                                0
Qualified Area Name                                                 0
Area (Land)                                                         0
Area (Water)           

In [11]:
# Check Duplicates
print(zhvi.duplicated().sum())
print(zori.duplicated().sum())
print(population.duplicated().sum())
print(unemployment.duplicated().sum())

0
0
0
0


**Interpretation**: The population and unemployment files have 59 rows, which is the 58 counties plus that code row. ZHVI has no missing values, but ZORI's gaps shrink toward the latest month (224 missing in April 2026, only 1 in August 2026), so the missing rents mostly come from series that start late rather than counties dropping out. None of the four files has duplicate rows.

#### Data Cleaning

Transform both zillow dataset into long format *ZHVI* & *ZORI* and filtered the dates to include only records from 2020 to 2025.

In [12]:
START, END = "2020-01", "2026-01"

def zillow_to_long(data, value_name, start=START, end=END):

    cols = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "StateCodeFIPS", "MunicipalCodeFIPS"]
    dates = [i for i in data.columns if i[0].isdigit() and start <= i < end]

    ca = data.loc[data["StateName"] == "CA", cols + dates].copy()
    # FIPS codes is 00+000
    ca["FIPS"] = (ca["StateCodeFIPS"].astype(str).str.zfill(2)
                  + ca["MunicipalCodeFIPS"].astype(str).str.zfill(3))

    ca["RegionName"] = ca["RegionName"].str.strip()

    # wide to long https://pandas.pydata.org/docs/reference/api/pandas.melt.html
    df = ca.melt(id_vars=["FIPS", "RegionName"], value_vars=dates,
                   var_name="Date", value_name=value_name)

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    return df.sort_values(["FIPS", "Date"]).reset_index(drop=True)

# Call the function
zhvi_CA = zillow_to_long(zhvi, "ZHVI")
zori_CA = zillow_to_long(zori, "ZORI")

# See the shape & check unique values
print("ZHVI:", zhvi_CA.shape, "| counties:", zhvi_CA["FIPS"].nunique(), "| months:", zhvi_CA["Date"].nunique())
print("ZORI:", zori_CA.shape, "| counties:", zori_CA["FIPS"].nunique(), "| months:", zori_CA["Date"].nunique())

ZHVI: (4176, 4) | counties: 58 | months: 72
ZORI: (3528, 4) | counties: 49 | months: 72


ZHVI covers all 58 counties × 72 months = 4,176 rows. ZORI has fewer counties because Zillow does not publish rents for some small rural counties.

In [13]:
# See the data types in the dataframe
zhvi_CA.dtypes

,0
FIPS,object
RegionName,object
Date,datetime64[ns]
ZHVI,float64


In [14]:
zori_CA.dtypes

,0
FIPS,object
RegionName,object
Date,datetime64[ns]
ZORI,float64


In [15]:
zhvi_CA.head()

,FIPS,RegionName,Date,ZHVI
0,06001,Alameda County,2020-01-31,842261.950
1,06001,Alameda County,2020-02-29,845568.279
2,06001,Alameda County,2020-03-31,849961.749
3,06001,Alameda County,2020-04-30,854051.748
4,06001,Alameda County,2020-05-31,853204.560


In [16]:
zori_CA.head()

,FIPS,RegionName,Date,ZORI
0,06001,Alameda County,2020-01-31,2547.423
1,06001,Alameda County,2020-02-29,2548.081
2,06001,Alameda County,2020-03-31,2563.948
3,06001,Alameda County,2020-04-30,2561.371
4,06001,Alameda County,2020-05-31,2556.497


### Data Manipulation

#### Merge 1: ZHVI x ZORI housing

In [17]:
housing = zhvi_CA.merge(
    zori_CA.drop(columns="RegionName"),
    on=["FIPS", "Date"],
    how="outer",            # keep every county-month that has a home value
    validate="1:1",
    indicator=True,
)

In [18]:
housing.query("_merge == 'left_only'")

,FIPS,RegionName,Date,ZHVI,ZORI,_merge
72,06003,Alpine County,2020-01-31,389374.972,NaN,left_only
73,06003,Alpine County,2020-02-29,389221.795,NaN,left_only
74,06003,Alpine County,2020-03-31,388438.388,NaN,left_only
75,06003,Alpine County,2020-04-30,388918.625,NaN,left_only
76,06003,Alpine County,2020-05-31,389024.193,NaN,left_only
...,...,...,...,...,...,...
3811,06105,Trinity County,2025-08-31,263132.164,NaN,left_only
3812,06105,Trinity County,2025-09-30,262568.495,NaN,left_only
3813,06105,Trinity County,2025-10-31,261993.268,NaN,left_only
3814,06105,Trinity County,2025-11-30,261540.047,NaN,left_only


In [19]:
# checks
assert len(housing) == len(zhvi_CA) == 58 * 72
assert housing[["FIPS", "Date"]].duplicated().sum() == 0

print(housing["_merge"].value_counts(), "\n")

_merge
both          3528
left_only      648
right_only       0
Name: count, dtype: int64 



In [20]:
no_rent = sorted(housing.loc[housing["_merge"] == "left_only", "RegionName"].unique())
print(f"{len(no_rent)} counties have no ZORI in any month:\n", no_rent)
housing = housing.drop(columns="_merge")

9 counties have no ZORI in any month:
 ['Alpine County', 'Colusa County', 'Lassen County', 'Mariposa County', 'Modoc County', 'Mono County', 'Plumas County', 'Sierra County', 'Trinity County']


**Interpretation**: The left join keeps all 4,176 county-months (58 x 72), so no home-value observation is lost. The 648 `left_only` rows are the 9 counties Zillow does not publish rents for at all (Alpine, Colusa, Lassen, Mariposa, Modoc, Mono, Plumas, Sierra, Trinity): small rural counties with few rental listings for a stable index. That is expected and is not a merge error.

#### Merge 2: Unemployment x Population

#### Population (from Social Explorer)

In [21]:
population.head()

,FIPS Code,Summary Level,Area Name,Qualified Area Name,"Total Population (July 1, 2025)"
0,Geo_FIPS,_geo_level_,Geo_NAME,Geo_qname,ORG_T007_001
1,06001,SL050,Alameda County,"Alameda County, California",1636630
2,06003,SL050,Alpine County,"Alpine County, California",1043
3,06005,SL050,Amador County,"Amador County, California",41876
4,06007,SL050,Butte County,"Butte County, California",209211


In [22]:
population.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 5 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   FIPS Code                        59 non-null     object
 1   Summary Level                    59 non-null     object
 2   Area Name                        59 non-null     object
 3   Qualified Area Name              59 non-null     object
 4   Total Population (July 1, 2025)  59 non-null     object
dtypes: object(5)
memory usage: 2.4+ KB


In [23]:
population = (population.iloc[1:,:][["FIPS Code", "Area Name", "Total Population (July 1, 2025)"]].rename(columns={"FIPS Code": "FIPS", "Area Name": "County", "Total Population (July 1, 2025)": "pop_2025"}))

print(population.shape)

population.head()

(58, 3)


,FIPS,County,pop_2025
1,06001,Alameda County,1636630
2,06003,Alpine County,1043
3,06005,Amador County,41876
4,06007,Butte County,209211
5,06009,Calaveras County,46605


In [24]:
population["pop_2025"] = population["pop_2025"].astype(int)

#### Unemployment (ACS 2020–2024 5-year Estimates)

In [25]:
# row 0 holds the (Geo_FIPS, SE_A17005_001, ...), so drop it
unemployment = (unemployment
                .loc[1:, ["FIPS", "NAME",
                          "Civilian Population in Labor Force 16 Years and Over",
                          "Civilian Population in Labor Force 16 Years and Over: Unemployed"]]
                .rename(columns={"NAME": "County",
                                 "Civilian Population in Labor Force 16 Years and Over": "labor_force",
                                 "Civilian Population in Labor Force 16 Years and Over: Unemployed": "unemployed"}))

unemployment["County"] = unemployment["County"].str.replace(", California", "")

unemployment["FIPS"] = unemployment["FIPS"].astype(str).str.zfill(5)

unemployment[["labor_force", "unemployed"]] = unemployment[["labor_force", "unemployed"]].astype(int)

unemployment["unemp_rate"] = unemployment["unemployed"] / unemployment["labor_force"] * 100

print(unemployment.shape)

unemployment.head(3)

(58, 5)


,FIPS,County,labor_force,unemployed,unemp_rate
1,06001,Alameda County,908995,48866,5.376
2,06003,Alpine County,837,37,4.421
3,06005,Amador County,17257,932,5.401


In [26]:
labor = unemployment.merge(population.drop(columns="County"), on="FIPS",
                           how="outer", validate="1:1", indicator=True)

print(labor["_merge"].value_counts())

labor = labor.drop(columns="_merge")

labor.head()

_merge
both          58
left_only      0
right_only     0
Name: count, dtype: int64


,FIPS,County,labor_force,unemployed,unemp_rate,pop_2025
0,06001,Alameda County,908995,48866,5.376,1636630
1,06003,Alpine County,837,37,4.421,1043
2,06005,Amador County,17257,932,5.401,41876
3,06007,Butte County,100743,7515,7.460,209211
4,06009,Calaveras County,19422,877,4.515,46605


**Interpretation**: All 58 counties matched on both sides, with no `left_only` or `right_only` rows. That's what I expected, since both files are Social Explorer exports for the same list of California counties and FIPS is kept as a 5-digit string in both.

In [27]:
labor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   FIPS         58 non-null     object 
 1   County       58 non-null     object 
 2   labor_force  58 non-null     int64  
 3   unemployed   58 non-null     int64  
 4   unemp_rate   58 non-null     float64
 5   pop_2025     58 non-null     int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 2.8+ KB


## Mapping: 1. Counties to Regions
Group into Six Regions

Source: https://coolcalifornia.arb.ca.gov/regions

In [28]:
regions = {
    "Bay Area": ["Alameda", "Contra Costa", "Marin", "Napa", "San Francisco",
                 "San Mateo", "Santa Clara", "Solano", "Sonoma"],
    "Southern California": ["Imperial", "Los Angeles", "Orange", "Riverside",
                            "San Bernardino", "San Diego", "Ventura"],
    "Central Coast": ["Monterey", "San Benito", "San Luis Obispo", "Santa Barbara", "Santa Cruz"],
    "Central Valley": ["Butte", "Colusa", "Fresno", "Glenn", "Kern", "Kings", "Madera",
                       "Merced", "Sacramento", "San Joaquin", "Stanislaus", "Sutter",
                       "Tehama", "Tulare", "Yolo", "Yuba"],
    "Sierra / Gold Country": ["Alpine", "Amador", "Calaveras", "El Dorado", "Inyo",
                              "Mariposa", "Mono", "Nevada", "Placer", "Plumas",
                              "Sierra", "Tuolumne"],
    "North Coast / Far North": ["Del Norte", "Humboldt", "Lake", "Lassen", "Mendocino",
                                "Modoc", "Shasta", "Siskiyou", "Trinity"],
}

county_to_region = {}

for region, county in regions.items():
  for c in county:
    county_to_region[f"{c} County"] = region

housing["region"] = housing["RegionName"].map(county_to_region)

housing.drop_duplicates("FIPS")["region"].value_counts() # Total should be 58

,count
region,
Central Valley,16
Sierra / Gold Country,12
Bay Area,9
North Coast / Far North,9
Southern California,7
Central Coast,5


**Interpretation**: Every county got a region, since the counts add up to 58 and nothing was left unmapped. The groups are pretty uneven, though: the Central Valley has 16 counties and the Central Coast only 5, so the averages for the smaller regions rest on just a handful of places.

## Mapping 2: population to county size

In [29]:
labor["size_tier"] = pd.cut(labor["pop_2025"],
                            bins=[0, 100_000, 1_000_000, np.inf],
                            labels=["Small (<100k)", "Mid (100k-1M)", "Large (1M+)"])
labor["size_tier"].value_counts().sort_index()

,count
size_tier,
Small (<100k),22
Mid (100k-1M),26
Large (1M+),10


**Interpretation**: Most California counties are small or mid-sized, and only 10 of the 58 have more than a million people. The 100k and 1M cutoffs are my own round numbers rather than an official definition, but they're easy to explain and leave each group with a reasonable number of counties.

# ProblemSet 2

#### Groupby/Agg 1: County-level housing growth

I first aggregate the monthly Zillow data to the county level. This allows me to compare home values and rents at the beginning and end of the study period.

The county-level growth measures directly address the first two research questions.

**!!! Note: I filtered the county that only has the complete 72 months of ZORI data, so that I can compare !!!!**

In [30]:
housing_full_zori_county = housing[~housing["ZORI"].isna()]

housing_full_zori_county = housing_full_zori_county.groupby(["FIPS", "RegionName"]).agg(months=("Date", "nunique"))

# This line of code I'm using Gemini to help me how to get the values from index
housing_full_zori_county = housing_full_zori_county[housing_full_zori_county["months"] == 72].index.get_level_values('RegionName').unique().to_list()

In [31]:
len(housing_full_zori_county)

29

**Interpretation**: Only 29 of the 58 counties have rent data for all 72 months, so the rent-versus-home-value comparison in RQ2 is based on these 29. Most of the dropped counties are small rural ones whose ZORI series starts late or doesn't exist at all, but Orange and San Francisco also got cut because they're missing one or two months in mid-2020. Whatever pattern shows up here mainly describes the larger, better-covered counties.

In [32]:
housing = housing[housing["RegionName"].isin(housing_full_zori_county)].reset_index(drop=True)

In [33]:
# Group monthly housing data by county
county_housing = (
    housing
    .sort_values(["FIPS", "Date"])
    .groupby(["FIPS", "RegionName"])
    .agg(
        zhvi_2020=("ZHVI", "first"),
        zhvi_2025=("ZHVI", "last"),
        zori_2020=("ZORI", "first"),
        zori_2025=("ZORI", "last"),
        months=("Date", "nunique")
    ).reset_index())

# Calculate percentage growth
county_housing["zhvi_growth_pct"] = ((county_housing["zhvi_2025"] - county_housing["zhvi_2020"])
                                      / county_housing["zhvi_2020"] * 100)

county_housing["zori_growth_pct"] = ((county_housing["zori_2025"] - county_housing["zori_2020"])
                                      / county_housing["zori_2020"] * 100)

county_housing.sort_values("zhvi_growth_pct", ascending=False).reset_index(drop=True).head(10)

,FIPS,RegionName,zhvi_2020,zhvi_2025,zori_2020,zori_2025,months,zhvi_growth_pct,zori_growth_pct
0,06083,Santa Barbara County,604363.958,970185.816,2111.031,3125.583,72,60.530,48.060
1,06029,Kern County,236887.162,362888.534,1139.586,1730.258,72,53.190,51.832
2,06073,San Diego County,606555.564,927598.249,2082.318,2912.131,72,52.929,39.850
3,06065,Riverside County,402042.996,605008.058,1740.369,2554.693,72,50.483,46.790
4,06071,San Bernardino County,369240.934,547407.809,1648.598,2412.321,72,48.252,46.326
5,06107,Tulare County,242725.451,358608.931,1124.410,1690.461,72,47.743,50.342
6,06019,Fresno County,278536.136,405709.400,1342.323,1988.009,72,45.658,48.102
7,06079,San Luis Obispo County,617020.411,891942.555,1955.495,2780.955,72,44.556,42.212
8,06085,Santa Clara County,1125886.109,1608192.584,3064.590,3540.084,72,42.838,15.516
9,06047,Merced County,295777.004,418573.042,1337.669,1995.053,72,41.516,49.144


**Interpretation**: Santa Barbara leads at 60.5%, followed by Kern, San Diego, and Riverside, all above 50%. What stands out is how many cheaper inland counties make the top ten (Kern, Tulare, Fresno, Merced), and in those places rents grew about as fast as home values, roughly 48–52%. Santa Clara is the odd one out: home values rose 43% but rent only 16%.

#### Groupby/Agg 2: Regional housing market

Next, I aggregate the county-level housing data by region. This makes it possible to compare broader geographic housing-market patterns rather than looking only at individual counties.

In [34]:
# Add the regional classification created earlier in the project
county_housing["region"] = county_housing["RegionName"].map(county_to_region)

regional_housing = (county_housing.groupby("region").agg(
        avg_zhvi_growth=("zhvi_growth_pct", "mean"),
        avg_zori_growth=("zori_growth_pct", "mean"),
        counties=("FIPS", "nunique")
    ).reset_index().sort_values("avg_zhvi_growth", ascending=False)
)

regional_housing

,region,avg_zhvi_growth,avg_zori_growth,counties
1,Central Coast,46.377,40.156,4
5,Southern California,45.524,38.622,5
2,Central Valley,37.585,40.864,9
4,Sierra / Gold Country,30.713,38.940,2
3,North Coast / Far North,30.065,44.241,1
0,Bay Area,25.489,18.269,8


**Interpretation**: The Central Coast and Southern California grew the fastest, around 46% on average, while the Bay Area trailed at 25% for home values and just 18% for rent. Keep the group sizes in mind after the filter: the North Coast row is really just one county, and Sierra / Gold Country is only two counties, so those two rows shouldn't be read as regional trends.

#### Recode 1: Categorizing home-value growth

To make differences in housing growth easier to compare, I recode the continuous home-value growth measure into three categories.

This classification distinguishes counties with relatively low, moderate, and high home-value growth during the study period.

In [35]:
# Use the 33rd and 67th percentiles to define relative growth categories
low_cutoff = county_housing["zhvi_growth_pct"].quantile(1/3) # 33rd percentiles
high_cutoff = county_housing["zhvi_growth_pct"].quantile(2/3) # 67th percentiles

county_housing["growth_category"] = pd.cut(
    county_housing["zhvi_growth_pct"],
    bins=[-np.inf, low_cutoff, high_cutoff, np.inf],
    labels=["Low growth", "Medium growth", "High growth"]
)

county_housing["growth_category"].value_counts().sort_index()

,count
growth_category,
Low growth,10
Medium growth,9
High growth,10


**Interpretation**: The 33rd/67th percentile cutoffs split the 29 counties into three almost equal groups (10, 9, and 10). The groups are relative, so "Low growth" doesn't mean prices fell: every county in the sample gained value, and even the slowest one, Butte, was up about 17%.

#### Recode 2: Recode months into housing-market periods

The monthly data contain many observations for each county. I recode the dates into three broader periods so that I can compare the beginning, middle, and end of the study period.

In [36]:
# Create a three-period variable from the monthly date
period_map = {
    2020: "Early period",
    2021: "Early period",
    2022: "Middle period",
    2023: "Middle period",
    2024: "Late period",
    2025: "Late period"
}

housing["housing_period"] = housing["Date"].dt.year.map(period_map)

housing[["FIPS", "RegionName", "Date", "ZHVI", "ZORI", "housing_period"]].head(10)

,FIPS,RegionName,Date,ZHVI,ZORI,housing_period
0,06001,Alameda County,2020-01-31,842261.950,2547.423,Early period
1,06001,Alameda County,2020-02-29,845568.279,2548.081,Early period
2,06001,Alameda County,2020-03-31,849961.749,2563.948,Early period
3,06001,Alameda County,2020-04-30,854051.748,2561.371,Early period
4,06001,Alameda County,2020-05-31,853204.560,2556.497,Early period
5,06001,Alameda County,2020-06-30,849121.942,2537.683,Early period
6,06001,Alameda County,2020-07-31,846140.420,2539.426,Early period
7,06001,Alameda County,2020-08-31,849274.700,2524.081,Early period
8,06001,Alameda County,2020-09-30,858836.795,2505.911,Early period
9,06001,Alameda County,2020-10-31,871430.855,2491.338,Early period


**Interpretation**: Each period covers two calendar years, so every county has 24 months in each one. The split roughly follows the pandemic boom (2020–21), the rate-hike years (2022–23), and the most recent stretch (2024–25), which is easier to compare than 72 separate months.

### Replace on Condition 1: Categorize unemployment conditions

I classify counties according to their unemployment rate. This allows me to compare housing growth across counties with different labor-market conditions.

In [37]:
labor["unemployment_status"] = "Low unemployment"

labor.loc[labor["unemp_rate"] >= 5, "unemployment_status"] = "Moderate unemployment"

labor.loc[labor["unemp_rate"] >= 8, "unemployment_status"] = "High unemployment"

labor[["County", "unemp_rate", "unemployment_status"]].sort_values("unemp_rate", ascending=False).head(10)

,County,unemp_rate,unemployment_status
45,Sierra County,12.631,High unemployment
12,Imperial County,12.522,High unemployment
16,Lake County,11.058,High unemployment
15,Kings County,10.803,High unemployment
23,Merced County,10.557,High unemployment
19,Madera County,10.119,High unemployment
5,Colusa County,9.776,High unemployment
53,Tulare County,9.312,High unemployment
31,Plumas County,9.152,High unemployment
22,Mendocino County,9.020,High unemployment


Sierra County tops the list at 12.6%, but its labor force is only about 1,200 people, so a handful of job losses can move its rate a lot. This table covers all 58 counties, though, and only Merced and Tulare from this top ten are in the 29-county housing sample.

### Subset 1: Large California counties

I also create a subset of large counties to examine whether housing-market patterns among the largest population centers differ from the statewide county-level pattern.

In [38]:
large_counties = labor[labor["size_tier"] == "Large (1M+)"].copy()

large_counties[["FIPS", "County", "pop_2025", "unemp_rate", "size_tier"]].sort_values("pop_2025", ascending=False)

,FIPS,County,pop_2025,unemp_rate,size_tier
18,06037,Los Angeles County,9694934,7.271,Large (1M+)
36,06073,San Diego County,3282248,6.110,Large (1M+)
29,06059,Orange County,3149507,5.404,Large (1M+)
32,06065,Riverside County,2544916,6.501,Large (1M+)
35,06071,San Bernardino County,2224091,7.109,Large (1M+)
42,06085,Santa Clara County,1914391,4.867,Large (1M+)
0,06001,Alameda County,1636630,5.376,Large (1M+)
33,06067,Sacramento County,1618460,6.712,Large (1M+)
6,06013,Contra Costa County,1170070,6.490,Large (1M+)
9,06019,Fresno County,1035456,8.693,Large (1M+)


**Interpretation**: These ten counties hold about 72% of California's population, so this small subset covers most of the people in the state. Fresno is the only one above 8% unemployment and Santa Clara the only one below 5%, so even the biggest counties face pretty different labor markets.

### Merge: County housing growth and local labor markets

This merge creates a single county-level dataset that combines housing-market growth with population size and local labor-market conditions.

In [39]:
final_county = county_housing.merge(
    labor[["FIPS", "pop_2025", "unemp_rate", "size_tier", "unemployment_status"]], on="FIPS", how="outer", validate="1:1", indicator=True)

final_county["_merge"].value_counts() # outer is good

final_county.drop(columns="_merge", inplace= True)

,count
_merge,
right_only,29
both,29
left_only,0


**Interpretation**: The 29 `both` rows are the counties in my housing sample, and the 29 `right_only` rows are counties that exist in the labor data but were dropped earlier for not having complete ZORI data, so this is expected and not a merge error. Because I used an outer join, those 29 counties stay in `final_county` with empty housing columns, so I'll need to filter them out before comparing growth with unemployment.

In [40]:
final_county.sort_values("zhvi_growth_pct", ascending=False).reset_index(drop=True).head()

,FIPS,RegionName,zhvi_2020,zhvi_2025,zori_2020,zori_2025,months,zhvi_growth_pct,zori_growth_pct,region,growth_category,pop_2025,unemp_rate,size_tier,unemployment_status
0,06083,Santa Barbara County,604363.958,970185.816,2111.031,3125.583,72.000,60.530,48.060,Central Coast,High growth,442065,6.418,Mid (100k-1M),Moderate unemployment
1,06029,Kern County,236887.162,362888.534,1139.586,1730.258,72.000,53.190,51.832,Central Valley,High growth,927068,8.627,Mid (100k-1M),High unemployment
2,06073,San Diego County,606555.564,927598.249,2082.318,2912.131,72.000,52.929,39.850,Southern California,High growth,3282248,6.110,Large (1M+),Moderate unemployment
3,06065,Riverside County,402042.996,605008.058,1740.369,2554.693,72.000,50.483,46.790,Southern California,High growth,2544916,6.501,Large (1M+),Moderate unemployment
4,06071,San Bernardino County,369240.934,547407.809,1648.598,2412.321,72.000,48.252,46.326,Southern California,High growth,2224091,7.109,Large (1M+),Moderate unemployment


**Interpretation**: The top five growers come in every size, from Santa Barbara (about 440k people) to San Diego (3.3M), and their unemployment ranges from 6.1% in San Diego to 8.6% in Kern. So at least among the fastest-growing counties, neither population nor unemployment lines up neatly with home-value growth.

# ProblemSet 3


### Home-value for all 58 counties

`zhvi_CA` from PS1 already has every county; only `housing` was cut to 29 in PS2. I copy it, add the region, and compute year-over-year growth for each county.

In [41]:
zhvi_CA = zillow_to_long(zhvi, "ZHVI", "2019-01", "2026-01")

In [42]:
home = zhvi_CA.copy()


home["region"] = home["RegionName"].map(county_to_region)
home["ym"] = home["Date"].dt.to_period("M")

home = home.sort_values(["FIPS", "Date"]).reset_index(drop=True)
home["zhvi_yoy"] = home.groupby("FIPS")["ZHVI"].pct_change(12, fill_method=None) * 100

# Keep only the original PS3 analysis period: 2020–2025.
home = home.loc[
    (home["Date"] >= "2020-01") &
    (home["Date"] < "2026-01")
].copy()

assert len(home) == 58 * 72
assert home["region"].notna().all()

print(home.shape, "| counties:", home["FIPS"].nunique(), "| months:", home["ym"].nunique())
print("missing YoY:", home["zhvi_yoy"].isna().sum())

# which home values are missing?
home.loc[home["ZHVI"].isna(), ["FIPS", "RegionName", "Date"]]

(4176, 7) | counties: 58 | months: 72
missing YoY: 4


,FIPS,RegionName,Date
1784,06043,Mariposa County,2020-09-30
1785,06043,Mariposa County,2020-10-31


**Interpretation**: 4,176 rows, and only two home values are missing: Mariposa in September and October 2020. Most of the 698 missing YoY values are just 2020 (58 × 12 = 696), since there's no earlier year to compare with; the other two are Mariposa's September and October 2021, which have no 2020 value to compare against.

## Merge 1: 30-year mortgage rate

30-Year Fixed Rate Mortgage Average in the United States, from Freddie Mac's weekly Primary Mortgage Market Survey. I downloaded the CSV from FRED and keep a copy in my repo.

Source: https://fred.stlouisfed.org/series/MORTGAGE30US

## Load the data

In [43]:
# Freddie Mac PMMS 30-year fixed rate, weekly, via FRED
mortgage_raw = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/MORTGAGE30US.csv")

print(mortgage_raw.shape)
print("missing:", mortgage_raw.isna().sum().sum(), "| duplicates:", mortgage_raw.duplicated().sum())
mortgage_raw.head()

(366, 2)
missing: 0 | duplicates: 0


,observation_date,MORTGAGE30US
0,2019-01-03,4.510
1,2019-01-10,4.450
2,2019-01-17,4.450
3,2019-01-24,4.450
4,2019-01-31,4.460


In [44]:
mortgage = mortgage_raw.rename(columns={"observation_date": "week", "MORTGAGE30US": "rate30"})

mortgage["week"] = pd.to_datetime(mortgage["week"])
mortgage["rate30"] = pd.to_numeric(mortgage["rate30"], errors="coerce")

print(mortgage.shape, "| missing rates:", mortgage["rate30"].isna().sum())

# PMMS is released on Thursdays, except some holiday weeks
mortgage["week"].dt.day_name().value_counts()

(366, 2) | missing rates: 0


,count
week,
Thursday,353
Wednesday,13


**Interpretation**: 353 of the 366 weekly readings fall on a Thursday and the other 13 on a Wednesday. Those are holiday weeks when Freddie Mac released a day early, which is one more reason not to match on the exact date.

#### Weekly to monthly

Zillow is monthly, so I average the 4-5 weekly observations in each month.

I also build two derived variables: the 12-month change in the rate, and a recode into a low-rate vs. high-rate regime.

In [45]:
mortgage_m = (mortgage
              .assign(ym = mortgage["week"].dt.to_period("M"))
              .groupby("ym", as_index=False)
              .agg(rate30=("rate30", "mean"),
                   n_weeks=("rate30", "count")))

# change vs. same month last year, in percentage points
mortgage_m["rate30_chg12"] = mortgage_m["rate30"] - mortgage_m["rate30"].shift(12)

# regime recode
mortgage_m["rate_regime"] = "Low rate (<5%)"
mortgage_m.loc[mortgage_m["rate30"] >= 5, "rate_regime"] = "High rate (>5%)"

# checks: one row per month, and every month has 4 or 5 weekly readings
assert mortgage_m["ym"].is_unique
assert mortgage_m["n_weeks"].between(4, 5).all()

print(mortgage_m.shape) # expect 84 months = 7 years x 12
mortgage_m["rate_regime"].value_counts()

(84, 5)


,count
rate_regime,
High rate (>5%),44
Low rate (<5%),40


In [46]:
mortgage_m.query("rate_regime == 'Low rate (<5%)'")

,ym,rate30,n_weeks,rate30_chg12,rate_regime
0,2019-01,4.464,5,NaN,Low rate (<5%)
1,2019-02,4.370,4,NaN,Low rate (<5%)
2,2019-03,4.265,4,NaN,Low rate (<5%)
3,2019-04,4.143,4,NaN,Low rate (<5%)
4,2019-05,4.072,5,NaN,Low rate (<5%)
5,2019-06,3.802,4,NaN,Low rate (<5%)
6,2019-07,3.765,4,NaN,Low rate (<5%)
7,2019-08,3.616,5,NaN,Low rate (<5%)
8,2019-09,3.605,4,NaN,Low rate (<5%)
9,2019-10,3.688,5,NaN,Low rate (<5%)


In [47]:
mortgage_m.query("rate_regime == 'High rate (>5%)'")

,ym,rate30,n_weeks,rate30_chg12,rate_regime
40,2022-05,5.230,4,2.267,High rate (>5%)
41,2022-06,5.522,5,2.547,High rate (>5%)
42,2022-07,5.412,4,2.544,High rate (>5%)
43,2022-08,5.223,4,2.380,High rate (>5%)
44,2022-09,6.112,5,3.212,High rate (>5%)
45,2022-10,6.900,4,3.833,High rate (>5%)
46,2022-11,6.805,4,3.737,High rate (>5%)
47,2022-12,6.364,5,3.266,High rate (>5%)
48,2023-01,6.272,4,2.827,High rate (>5%)
49,2023-02,6.258,4,2.495,High rate (>5%)


**Interpretation**: There are 84 months because I kept 2019 for the 12-month change. The 40 low-rate months run from January 2019 to April 2022, and from May 2022 on the rate never went back under 5%.

In [48]:
home_rate = home.merge(
    mortgage_m,
    on="ym",
    how="outer",          # outer on purpose, to see what doesn't match
    validate="m:1",       # many county-months to one national rate per month
    indicator=True,
)

home_rate["_merge"].value_counts()

,count
_merge,
both,4176
right_only,12
left_only,0


In [49]:
# crosstab the merge result by year
pd.crosstab(home_rate["ym"].dt.year, home_rate["_merge"])

_merge,right_only,both
ym,,
2019,12,0
2020,0,696
2021,0,696
2022,0,696
2023,0,696
2024,0,696
2025,0,696


**Interpretation**: Every county-month found a rate (696 per year = 58 × 12), so nothing on the housing side is lost. The 12 `right_only` rows are the 2019 months that I kept only for the 12-month change, and the Zillow data starts in 2020, so they're expected.

In [50]:
home_rate = (home_rate[home_rate["_merge"] == "both"]
             .drop(columns=["_merge", "n_weeks"])
             .reset_index(drop=True))

In [51]:
home_rate

,FIPS,RegionName,Date,ZHVI,region,ym,zhvi_yoy,rate30,rate30_chg12,rate_regime
0,06001,Alameda County,2020-01-31,842261.950,Bay Area,2020-01,-1.179,3.624,-0.840,Low rate (<5%)
1,06003,Alpine County,2020-01-31,389374.972,Sierra / Gold Country,2020-01,1.952,3.624,-0.840,Low rate (<5%)
2,06005,Amador County,2020-01-31,348422.225,Sierra / Gold Country,2020-01,2.342,3.624,-0.840,Low rate (<5%)
3,06007,Butte County,2020-01-31,342651.808,Central Valley,2020-01,11.982,3.624,-0.840,Low rate (<5%)
4,06009,Calaveras County,2020-01-31,335227.783,Sierra / Gold Country,2020-01,3.384,3.624,-0.840,Low rate (<5%)
...,...,...,...,...,...,...,...,...,...,...
4171,06107,Tulare County,2025-12-31,358608.931,Central Valley,2025-12,1.003,6.190,-0.525,High rate (>5%)
4172,06109,Tuolumne County,2025-12-31,388426.379,Sierra / Gold Country,2025-12,-3.783,6.190,-0.525,High rate (>5%)
4173,06111,Ventura County,2025-12-31,868142.827,Southern California,2025-12,-1.228,6.190,-0.525,High rate (>5%)
4174,06113,Yolo County,2025-12-31,621672.921,Central Valley,2025-12,-1.776,6.190,-0.525,High rate (>5%)


### Descriptive stats: housing-cost growth vs. the mortgage rate

Median county growth for each month, with the rate on the second axis. 2020 has no YoY value, so the chart starts in January 2021.

In [52]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

state_month = (home_rate.groupby("Date", as_index=False).agg(zhvi_yoy=("zhvi_yoy", "median"),
                                                         rate30=("rate30", "first")).dropna(subset=["zhvi_yoy"]))

fig = (make_subplots(specs=[[{"secondary_y": True}]])
       .add_trace(go.Scatter(x=state_month["Date"], y=state_month["zhvi_yoy"],
                             name="Home value YoY % (median of 58 counties)"), secondary_y=False)
       .add_trace(go.Scatter(x=state_month["Date"], y=state_month["rate30"],
                             name="30-yr mortgage rate %", line=dict(dash="dot", color="black")),
                  secondary_y=True)
       .add_hline(y=0, line_width=1, line_color="gray")
       .update_layout(title="California home-value growth vs. the 30-year mortgage rate, 2021-2025",
                      hovermode="x unified", legend=dict(orientation="h", y=-0.2))
       .update_yaxes(title_text="Year-over-year growth (%)", secondary_y=False)
       .update_yaxes(title_text="Mortgage rate (%)", secondary_y=True))

fig.show()

**Interpretation**:

In [53]:
regime_region = (home_rate
                 .dropna(subset=["zhvi_yoy"])
                 .groupby(["region", "rate_regime"])["zhvi_yoy"]
                 .mean()
                 .unstack("rate_regime")
                 .round(1))

regime_region["slowdown"] = regime_region["High rate (>5%)"] - regime_region["Low rate (<5%)"]

regime_region.sort_values("slowdown")

rate_regime,High rate (>5%),Low rate (<5%),slowdown
region,,,
North Coast / Far North,-0.800,10.200,-11.000
Sierra / Gold Country,1.000,12.000,-11.000
Central Valley,2.400,12.200,-9.800
Central Coast,3.000,11.900,-8.900
Bay Area,0.700,8.800,-8.100
Southern California,4.800,11.600,-6.800


**Interpretation**:

## Merge 2: FEMA National Risk Index

The National Risk Index (NRI) is FEMA's county-level estimate of exposure to 18 natural hazards. For each hazard it gives an expected annual loss in dollars, plus scores and ratings that are percentiles against every county in the U.S. The file I use is the December 2025 version.

Source: https://www.fema.gov/about/openfema/data-sets/national-risk-index-data

In [54]:
# FEMA National Risk Index, county table
nri = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/NRI_Table_Counties.csv")

print(nri.shape)
print("states/territories:", nri["STATEABBRV"].nunique())

(3232, 465)
states/territories: 56


In [55]:
# { STCOFIPS: State-County FIPS Code,
#   COUNTY: County Name,
#   POPULATION: Population (2020),
#   AREA: Area (sq mi),
#   BUILDVALUE: Building Value ($),
#   RISK_SCORE: National Risk Index - Score - Composite,
#   WFIR_EALB: Wildfire - Expected Annual Loss - Building Value,
#   WFIR_RISKS: Wildfire - Hazard Type Risk Index Score,
#   WFIR_RISKR: Wildfire - Hazard Type Risk Index Rating
# }

'\n{ STCOFIPS: State-County FIPS Code,\n  COUNTY: County Name,\n  POPULATION: Population (2020),\n  AREA: Area (sq mi),\n  BUILDVALUE: Building Value ($),\n  RISK_SCORE: National Risk Index - Score - Composite,\n  WFIR_EALB: Wildfire - Expected Annual Loss - Building Value,\n  WFIR_RISKS: Winter Weather - Hazard Type Risk Index Score,\n  WFIR_RISKR: Winter Weather - Hazard Type Risk Index Rating\n}\n'

In [56]:
cols = ["STCOFIPS", "COUNTY", "POPULATION", "AREA", "BUILDVALUE",
        "RISK_SCORE", "WFIR_EALB", "WFIR_RISKS", "WFIR_RISKR"]

nri_ca = nri.loc[nri["STATEABBRV"] == "CA", cols].copy()

print(nri_ca.shape)

nri_ca.info()

(58, 9)
<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 187 to 244
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   STCOFIPS    58 non-null     int64  
 1   COUNTY      58 non-null     object 
 2   POPULATION  58 non-null     int64  
 3   AREA        58 non-null     float64
 4   BUILDVALUE  58 non-null     float64
 5   RISK_SCORE  58 non-null     float64
 6   WFIR_EALB   58 non-null     float64
 7   WFIR_RISKS  58 non-null     float64
 8   WFIR_RISKR  58 non-null     object 
dtypes: float64(5), int64(2), object(2)
memory usage: 4.5+ KB


In [63]:
nri_ca["STCOFIPS"].head()

,STCOFIPS
187,6001
188,6003
189,6005
190,6007
191,6009


In [58]:
# Add the leading zero
nri_ca["FIPS"] = nri_ca["STCOFIPS"].astype(str).str.zfill(5)

In [59]:
print("counties with wildfire score >= 90:", (nri_ca["WFIR_RISKS"] >= 90).sum(), "of", len(nri_ca))

nri_ca[["RISK_SCORE", "WFIR_RISKS"]].describe().round(1)

counties with wildfire score >= 90: 49 of 58


,RISK_SCORE,WFIR_RISKS
count,58.000,58.000
mean,88.700,94.200
std,17.500,10.500
min,10.600,28.700
25%,86.400,93.300
50%,94.800,97.800
75%,98.800,99.100
max,100.000,100.000


**Interpretation**: The scores are percentiles against the whole country, and 49 of the 58 California counties score 90 or higher on wildfire (median 97.8). So the score mostly says that California burns, not which counties burn more.

In [60]:
nri_ca["density"] = nri_ca["POPULATION"] / nri_ca["AREA"]   # people per square mile

(nri_ca
 .sort_values("WFIR_RISKS", ascending=False)
 [["FIPS", "COUNTY", "WFIR_RISKS", "WFIR_RISKR", "density"]]
 .head(10))

,FIPS,COUNTY,WFIR_RISKS,WFIR_RISKR,density
223,06073,San Diego,100.000,Very High,720.986
219,06065,Riverside,99.968,Very High,327.627
205,06037,Los Angeles,99.936,Very High,2085.660
222,06071,San Bernardino,99.905,Very High,107.344
216,06059,Orange,99.809,Relatively High,3318.703
242,06111,Ventura,99.777,Relatively High,378.285
201,06029,Kern,99.746,Relatively High,110.160
195,06017,El Dorado,99.714,Relatively High,105.841
231,06089,Shasta,99.491,Relatively High,46.811
191,06009,Calaveras,99.459,Relatively High,43.190


**Interpretation**:

In [61]:
home_risk = home_rate.merge(
    nri_ca.drop(columns="STCOFIPS"),
    on="FIPS",
    how="outer",
    validate="m:1",
    indicator=True,
)

home_risk["_merge"].value_counts()

,count
_merge,
both,4176
left_only,0
right_only,0


In [62]:
home_risk = home_risk.drop(columns="_merge")

home_risk.shape

(4176, 19)

**Interpretation**: Everything matched: 4,176 `both` and no `left_only` or `right_only`. That's expected here because both files cover the same full list of 58 California counties.

## Reference

1. Zillow Home Value Index (ZHVI), county, all homes, mid tier (35th–65th pct), smoothed, seasonally adjusted, monthly: https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv
   (Zillow Research data page: https://www.zillow.com/research/data/  *Home Values* Data Type "ZHVI All Homes (SFR, Condo/Co-op) Time Series, Smoothed, Seasonally Adjusted", Geography "County")

2. Zillow Observed Rent Index (ZORI), county, all homes plus multifamily, smoothed, monthly: https://files.zillowstatic.com/research/public_csvs/zori/County_zori_uc_sfrcondomfr_sm_month.csv

3. U.S. Census Bureau, American Community Survey 2020-2024 5-Year Estimates, table A17005 (Employment Status for Total Population 16 Years and Over), CA counties, via Social Explorer: https://www.socialexplorer.com/reports/socialexplorer/en/report/bac15980-b308-11f1-9512-c38da70940e6 — exported file: https://github.com/HowardChen0211/gisPy/raw/refs/heads/main/CA_UnemploymentRate.csv

4. California Air Resources Board, Cool California regions (county-to-region grouping): https://coolcalifornia.arb.ca.gov/regions

5. Freddie Mac, 30-Year Fixed Rate Mortgage Average in the United States [MORTGAGE30US], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/MORTGAGE30US, September 30, 2026.

6. Freddie Mac, 30-Year Fixed Rate Mortgage Average in the United States [MORTGAGE30US], Primary Mortgage Market Survey, weekly, retrieved from FRED, Federal Reserve Bank of St. Louis: https://fred.stlouisfed.org/series/MORTGAGE30US  (CSV loaded in the notebook: https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US ; original source: https://www.freddiemac.com/pmms)
